# Information
This notebook defines the necessary preprocessing steps to prepare the player_stats dataset for model training. A subset of these steps will later be exported into a standalone Python module and integrated into the data loading and preprocessing pipeline used for inference. These reusable steps are explicitly labeled throughout the notebook.

The remaining steps are specific to the data integration process and are required to merge data from Transfermarkt (TM) with data from Sofascore. This integration is necessary to link player_stats with the corresponding player ratings (grades) and is not part of the final preprocessing pipeline used during model deployment.

In [25]:
import pandas as pd
import numpy as np

tm_stats = pd.read_csv("../data/scrape/pro/player_stats.csv")
tm_matches = pd.read_csv("../data/scrape/pro/matches.csv")
tm_players = pd.read_csv("../data/scrape/pro/players.csv")

tm_players = tm_players[["player_id", "player_name", "position"]]
tm_matches = tm_matches[["match_id", "date"]]

tm_stats = tm_stats.merge(tm_players, on="player_id", how="left")
tm_stats = tm_stats.merge(tm_matches, on="match_id", how="left")

In [26]:
# Merge with the dataset from sofascore to get the grade
tm_stats["rating"] = np.random.uniform(1, 10, len(tm_stats)).round(1) #Temporary
# After the merge drop the unneccesary colums
cols = ["player_id", "match_id", "club_id", "player_name", "date"]
tm_stats = tm_stats.drop(columns = cols, errors="ignore")
df = tm_stats

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["goals", "assists", "minutes", "team_goals", "team_conceded"]
cat_cols = ["position"]
bool_cols = ["yellow", "red", "start_eleven"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

X = df.drop(columns=["rating"])
y = df["rating"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42     # reproduzierbar
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 2.254900829081633
RMSE: 2.6026966987679647
